In [ ]:
%load_ext autoreload
%autoreload 2

# pi-metaboqc Interactive Pipeline Tutorial

This notebook provides a step-by-step interactive guide to the `pi-metaboqc` metabolomics data quality control pipeline. 

By executing each cell sequentially, you can inspect the intermediate data structures and quality assessment (QA) reports generated at each stage of the workflow.

## Step 0: Environment Setup

First, we will import the core modules of `pi-metaboqc`, configure the input data paths, and create a dedicated output directory for the results of this notebook.

In [ ]:
# -------------------------------------------------------------------------
# Step 0: Environment Setup and Parameter Loading
# -------------------------------------------------------------------------
import os
import sys
import pandas as pd
from loguru import logger

import pimqc.io_utils as iu 
import pimqc.plot_utils as pu
from pimqc.dataset_builder import build_dataset
from pimqc.assessment import MetaboIntAssessor
from pimqc.filtering import MetaboIntFilter
from pimqc.correction import MetaboIntCorrector
from pimqc.imputation import MetaboIntImputer
from pimqc.normalization import MetaboIntNormalizer

logger.info("Environment initialized successfully.")

# Define standard directories
DATA_DIR = os.path.join("..", "src", "pimqc", "data")
OUTPUT_DIR = os.path.join(".", "tutorial_output")
iu._check_dir_exists(dir_path=OUTPUT_DIR, handle="makedirs")

# Load SSOT pipeline parameters
PARAMS_PATH = os.path.join(DATA_DIR, "pipeline_parameters.json")
params = iu._load_json_file(input_file_path=PARAMS_PATH)

# Load raw matrices
# meta_df = pd.read_csv(
#     os.path.join(DATA_DIR, "project_meta.csv"), header=[0])
meta_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_meta_with_simu_group.csv"), header=[0])

int_df = pd.read_csv(
    os.path.join(DATA_DIR, "project_intensity.csv"), index_col=[0], header=[0])

## Step 1: Dataset Construction
We will merge the raw intensity dataframe with the metadata mapping to 
build the standardized `MetaboInt` base object. This multi-indexed dataframe 
ensures analytical features and sample metadata are strictly aligned.

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Build Standardized Dataset
# -------------------------------------------------------------------------
step1_dir = os.path.join(OUTPUT_DIR, "01_Raw_Data")

raw_data = build_dataset(
    meta_info=meta_df,
    int_df=int_df,
    pipeline_params=params,
    mode=params.get("MetaboInt", {}).get("mode", "POS"),
    batch=params.get("MetaboInt", {}).get("batch", "Batch"),
    sample_type=params.get("MetaboInt", {}).get("sample_type", "Sample Type"),
    bio_group=params.get("MetaboInt", {}).get("bio_group", "Bio Group"),
    sample_name=params.get("MetaboInt", {}).get("sample_name", "Sample Name"),
    inject_order=params.get("MetaboInt", {}).get(
        "inject_order", "Inject Order"),
    output_dir=step1_dir
)

## Step 2: Quality Assessment (Raw Data)
Before any mathematical transformation, we perform an initial QA to evaluate 
the baseline condition of the untargeted metabolomics data.

In [ ]:
# -------------------------------------------------------------------------
# Step 2: Initial QA
# -------------------------------------------------------------------------
step2_dir = os.path.join(OUTPUT_DIR, "02_QA_Raw_Data")

qa_raw = MetaboIntAssessor(data=raw_data, pipeline_params=params)
qa_raw.execute_assessment(output_dir=step2_dir)

## Step 3: High Missing Value (MV) Feature Filter (Stage 1)
We drop features containing an unacceptably high proportion of missing 
values based on the defined `mv_group_tol` biological override rule.

In [ ]:
# -------------------------------------------------------------------------
# Step 3: Stage 1 Filtering & Missing Mechanism Classification
# -------------------------------------------------------------------------
step3_dir = os.path.join(OUTPUT_DIR, "03_Filtered_Stage1")

fltr_stg1 = MetaboIntFilter(data=raw_data, pipeline_params=params) 
mv_filtered_data = fltr_stg1.execute_mv_filtering(output_dir=step3_dir)

## Step 4: Quality Assessment on MV-Filtered Data


In [ ]:
# -------------------------------------------------------------------------
# Step 4: QA Post MV-Filtering
# -------------------------------------------------------------------------
step4_dir = os.path.join(OUTPUT_DIR, "04_QA_MV_Filtered_Data")

qa_fltr_stg1 = MetaboIntAssessor(data=mv_filtered_data, pipeline_params=params)
qa_fltr_stg1.execute_assessment(output_dir=step4_dir)

## Step 5: Signal Drift & Batch Effect Correction
Using Quality Control (QC) samples, we build regression models (e.g., LOESS, 
Random Forest, SVR) to estimate and correct analytical signal drift 
across injection sequences.

In [ ]:
# -------------------------------------------------------------------------
# Step 5: Signal Correction
# -------------------------------------------------------------------------
step5_dir = os.path.join(OUTPUT_DIR, "05_Corrected_Data")

sc_engine = MetaboIntCorrector(data=mv_filtered_data, pipeline_params=params)
intra_data, inter_data = sc_engine.execute_signal_correction(output_dir=step5_dir)

## Step 6 & 7: Quality Assessment on Corrected Data
We assess the effectiveness of the signal correction algorithms by examining 
the intra-batch and inter-batch states separately.

In [ ]:
# -------------------------------------------------------------------------
# Step 6 & 7: QA on Intra and Inter-batch Corrected Data
# -------------------------------------------------------------------------
step6_dir = os.path.join(OUTPUT_DIR, "06_QA_Intra_SC")

qa_intra = MetaboIntAssessor(data=intra_data, pipeline_params=params) 
qa_intra.execute_assessment(output_dir=step6_dir)

step7_dir = os.path.join(OUTPUT_DIR, "07_QA_Inter_SC")

qa_inter = MetaboIntAssessor(data=inter_data, pipeline_params=params)
qa_inter.execute_assessment(output_dir=step7_dir)

## Step 8: Low Quality Feature Filter (Stage 2)
Features are evaluated against Blank sample ratios and QC Relative Standard 
Deviation (RSD). Missing indices are passed explicitly as current features 
default to MAR behavior prior to imputation.

In [ ]:
# -------------------------------------------------------------------------
# Step 8: Stage 2 Filtering
# --------------------------------------- ----------------------------------
step8_dir = os.path.join(OUTPUT_DIR, "08_Filtered_Stage2")

fltr_stg2 = MetaboIntFilter(data=inter_data, pipeline_params=params)
quality_filtered_data = fltr_stg2.execute_quality_filtering(output_dir=step8_dir)

## Step 9: Quality Assessment on Quality-Filtered Data

In [ ]:
# -------------------------------------------------------------------------
# Step 9: QA Post Quality-Filtering
# -------------------------------------------------------------------------
step9_dir = os.path.join(OUTPUT_DIR, "09_QA_Quality_Filtered_Data")

qa_fltr = MetaboIntAssessor(data=quality_filtered_data, pipeline_params=params)
qa_fltr.execute_assessment(output_dir=step9_dir)

## Step 10: Missing Value Imputation
We address remaining structural missing values using probabilistic logic, 
Random Forest, or traditional k-Nearest Neighbors (KNN).

In [ ]:
# -------------------------------------------------------------------------
# Step 10: Missing Value Imputation
# -------------------------------------------------------------------------
step10_dir = os.path.join(OUTPUT_DIR, "10_Imputation")

# 1. Initialize the imputation engine
# The engine now incorporates a biologically-aware evaluation framework
imp_engine = MetaboIntImputer(data=quality_filtered_data, pipeline_params=params)

# 2. Execute missing value imputation in 'auto' mode
# Under the hood, the engine will automatically:
#   - Generate an abundance-dependent hybrid mask (simulating both MCAR and MNAR).
#   - Benchmark multiple algorithms (probabilistic, knn, global_min, column_min).
#   - Select the optimal algorithm by minimizing 'NRMSE_Low' (prioritizing LOD fidelity).
#   - Export the fully imputed object and stratified visual grids to the disk.
imputed_data = imp_engine.execute_imputation(method="auto", output_dir=step10_dir)

## Step 11: Quality Assessment on Imputed Data

In [ ]:
# -------------------------------------------------------------------------
# Step 11: QA Post-Imputation
# -------------------------------------------------------------------------
step11_dir = os.path.join(OUTPUT_DIR, "11_QA_Imputed_Data")

qa_imp = MetaboIntAssessor(data=imputed_data, pipeline_params=params)
qa_imp.execute_assessment(output_dir=step11_dir)

## Step 12: Data Normalization
Features are scaled and normalized (e.g., PQN, VSN) to remove remaining 
systematic bias and stabilize variance across the detection range.

In [ ]:
# -------------------------------------------------------------------------
# Step 12: Normalization
# -------------------------------------------------------------------------
step12_dir = os.path.join(OUTPUT_DIR, "12_Normalized_Data")

# 1. Initialize normalization engine (norm_engine serves as the Raw Data)
norm_engine = MetaboIntNormalizer(imputed_data, pipeline_params=params)
# 2. Execute normalization (this automatically saves comparison PDFs to disk)
col_data, norm_data = norm_engine.execute_normalization(output_dir=step12_dir)

## Step 13 & 14: Quality Assessment on Normalized Data

In [ ]:
step13_dir = os.path.join(OUTPUT_DIR, "13_QA_Column_Normalized_Data")

qa_col_norm = MetaboIntAssessor(data=col_data, pipeline_params=params)
qa_col_norm.execute_assessment(output_dir=step13_dir)

step14_dir = os.path.join(OUTPUT_DIR, "14_QA_Final_Normalized_Data")

qa_col_norm = MetaboIntAssessor(data=norm_data, pipeline_params=params)
qa_col_norm.execute_assessment(output_dir=step14_dir) 

logger.success("Pipeline tutorial completed successfully.")

## Step 15: Merge Quality Assessment plots into one summary, and generate report markdown

In [ ]:
from pimqc import report_utils as ru

reporter = ru.ProjectReporter(base_dir=OUTPUT_DIR)
reporter.compile_assessor_report(cols=4, report_folder="15_Report_Markdown")